# Analyse des protéines de liaison à l'actine (ABP)

Ce notebook explore les interactions actine–ABP à partir des données filtrées :
- Vue d'ensemble des ABP (nombre d'interactions, sites S1, clusters C70)
- Analyse de compétition : quels ABPs partagent le même site S1 sur l'actine
- Heatmap de présence ABP × site S1
- Groupes de compétition et comparaison des interfaces
- Stats PDB sans ABP (filaments d'actine pure)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

DATA = Path("../data/filtered")

df_all   = pd.read_csv(DATA / "filtered_all_data.csv")
df_pp    = pd.read_csv(DATA / "proteins_per_pdb.csv")
df_c70   = pd.read_csv(DATA / "patches_infos_cluster_data_70.csv")
df_s1    = pd.read_csv(DATA / "patches_infos_s1_binding_site.csv")

df_all["s1_actine"] = df_all["s1_actine"].fillna(False)
df_all["s2_actine"] = df_all["s2_actine"].fillna(False)
df_all["_pdb"]      = df_all["subunit_1"].str.split("_").str[0]

print(f"{len(df_all)} interactions totales · {df_all['_pdb'].nunique()} PDB")

2074 interactions totales · 145 PDB


## 1. Vue d'ensemble des ABP

In [2]:
# Interactions hétéro : actin S1 ↔ ABP S2
hetero = df_all[df_all["s1_actine"] & ~df_all["s2_actine"]].copy()

# Lier avec proteins_per_pdb pour avoir le nom propre de la protéine
df_abp    = df_pp[~df_pp["is_actin"]].copy()
df_abp["chain_low"] = df_abp["chain"].str.lower()
hetero["subunit_2_low"] = hetero["subunit_2"].str.lower()
hetero_m  = hetero.merge(df_abp[["chain_low", "protein", "pdb_id"]],
                         left_on="subunit_2_low", right_on="chain_low", how="left")

abp_stats = (
    hetero_m.groupby("protein")
    .agg(
        nb_interactions=("subunit_1", "count"),
        nb_pdb=("_pdb", "nunique"),
        nb_c70=("cluster_data_70", "nunique"),
        nb_s1=("s1_binding_site_cluster_data_70", "nunique"),
        s1_sites=("s1_binding_site_cluster_data_70",
                  lambda x: ", ".join(sorted(x.dropna().astype(str).unique()))),
    )
    .reset_index()
    .sort_values("nb_interactions", ascending=False)
)

print(f"{len(abp_stats)} ABPs distincts · {len(hetero)} interactions hétéro")
abp_stats.head(15)

54 ABPs distincts · 800 interactions hétéro


,protein,nb_interactions,nb_pdb,nb_c70,nb_s1,s1_sites
17,"Coronin-1B,Methylated-DNA--protein-cysteine me...",91,6,6,7,"6685_260, 6685_261, 6685_262, 6685_263, 6685_2..."
16,Cofilin-2,72,10,2,2,"6685_0, 6685_21"
15,Cofilin-1,64,9,2,3,"6685_0, 6685_200, 6685_21"
14,Cell invasion protein SipA,61,4,14,11,"6685_154, 6685_178, 6685_179, 6685_180, 6685_1..."
43,Tropomyosin alpha-1 chain,35,7,17,9,"6685_128, 6685_129, 6685_130, 6685_135, 6685_1..."
51,VopV,34,1,3,3,"6685_257, 6685_258, 6685_259"
39,Spectrin beta chain,32,2,2,2,"6685_121, 6685_141"
49,Vibrio VopV,31,1,3,3,"6685_257, 6685_258, 6685_259"
25,Inverted formin-2,30,5,22,18,"6685_0, 6685_200, 6685_202, 6685_209, 6685_210..."
13,Catenin alpha-1,28,4,4,4,"6685_0, 6685_102, 6685_141, 6685_16"


In [3]:
# Bar chart : top ABPs par nb interactions
top20 = abp_stats.head(20).sort_values("nb_interactions")
fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(top20["protein"], top20["nb_interactions"], color="#4a90d9", height=0.7)
ax.bar_label(bars, fmt="%d", padding=3, fontsize=8)
ax.set_xlabel("Nb interactions")
ax.set_title("Top 20 ABPs — nb interactions actine–ABP")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

/var/folders/hv/x060xf8501g94gz9j0y0wqxm0000gn/T/ipykernel_7797/1419469279.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Sites S1 utilisés par chaque ABP

In [4]:
# Matrice binaire ABP × site S1 (présence = au moins 1 interaction)
abp_s1 = (
    hetero_m.dropna(subset=["protein", "s1_binding_site_cluster_data_70"])
    .groupby(["protein", "s1_binding_site_cluster_data_70"])["subunit_1"]
    .count()
    .rename("nb_inter")
    .reset_index()
)

# Pivoter : lignes = ABP, colonnes = site S1
mat = abp_s1.pivot_table(
    index="protein",
    columns="s1_binding_site_cluster_data_70",
    values="nb_inter",
    aggfunc="sum",
    fill_value=0,
)

# Trier ABPs par nb de sites, sites par nb d'ABPs
mat = mat.loc[
    mat.gt(0).sum(axis=1).sort_values(ascending=False).index,
    mat.gt(0).sum(axis=0).sort_values(ascending=False).index,
]

print(f"Matrice : {mat.shape[0]} ABPs × {mat.shape[1]} sites S1")
print(f"Sites S1 avec 2+ ABPs : {(mat.gt(0).sum() >= 2).sum()}")

Matrice : 54 ABPs × 143 sites S1
Sites S1 avec 2+ ABPs : 29


In [5]:
# Heatmap ABP × site S1 (présence/absence + intensité = nb interactions)
mat_bool = mat.gt(0).astype(int)  # pour le masque
mat_log  = np.log1p(mat)          # log pour atténuer les grands écarts

# Ne garder que les sites partagés par ≥ 2 ABPs
shared_sites = mat.columns[mat.gt(0).sum() >= 2]
mat_shared   = mat_log[shared_sites]

fig_h = max(6, len(mat_shared) * 0.28)
fig_w = max(8, len(shared_sites) * 0.55)
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
sns.heatmap(
    mat_shared,
    ax=ax,
    cmap="YlOrRd",
    linewidths=0.3,
    linecolor="#dddddd",
    mask=mat_shared == 0,
    cbar_kws={"label": "log(nb interactions + 1)"},
)
ax.set_title("ABP × site S1 (sites partagés par ≥ 2 ABPs)", fontsize=12)
ax.set_xlabel("Site S1 (cluster de liaison actine)")
ax.set_ylabel("ABP")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

/var/folders/hv/x060xf8501g94gz9j0y0wqxm0000gn/T/ipykernel_7797/2456179832.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Analyse de compétition — sites S1 partagés

In [6]:
# Pour chaque site S1 : liste des ABPs qui s'y fixent
competition = (
    hetero_m.dropna(subset=["protein", "s1_binding_site_cluster_data_70"])
    .groupby("s1_binding_site_cluster_data_70")
    .agg(
        nb_abp=("protein", "nunique"),
        abps=("protein", lambda x: sorted(x.unique().tolist())),
        nb_inter=("subunit_1", "count"),
        nb_pdb=("_pdb", "nunique"),
    )
    .reset_index()
    .rename(columns={"s1_binding_site_cluster_data_70": "Site S1"})
    .sort_values("nb_abp", ascending=False)
)

compet_multi = competition[competition["nb_abp"] >= 2].copy()
compet_multi["ABPs"] = compet_multi["abps"].apply(lambda x: " · ".join(x))

print(f"{len(compet_multi)} sites S1 avec ≥ 2 ABPs (compétition potentielle)")
compet_multi[["Site S1", "nb_abp", "nb_inter", "nb_pdb", "ABPs"]]

29 sites S1 avec ≥ 2 ABPs (compétition potentielle)


,Site S1,nb_abp,nb_inter,nb_pdb,ABPs
131,6685_6,6,19,11,"Myosin-14,Alpha-actinin A · Myosin-6 · Myosin-..."
0,6685_0,5,83,22,Alpha-catenin-like protein hmp-1 · Catenin alp...
13,6685_121,4,23,5,Inositol-trisphosphate 3-kinase A · Plastin-3 ...
42,6685_16,4,22,5,Catenin alpha-1 · Epididymis secretory protein...
31,6685_15,4,16,4,Actin-related protein 2/3 complex subunit 1B ·...
1,6685_102,3,9,4,Catenin alpha-1 · Inositol-trisphosphate 3-kin...
68,6685_193,3,7,4,Actin-related protein 2/3 complex subunit 1 · ...
27,6685_144,3,9,5,Actin-related protein 2/3 complex subunit 1 · ...
24,6685_141,3,28,5,Alpha-catenin-like protein hmp-1 · Catenin alp...
71,6685_200,3,8,4,Cofilin-1 · Inverted formin-2 · Protein diapha...


In [7]:
# Bar chart : sites S1 les plus "disputés"
top_sites = compet_multi.head(15).sort_values("nb_abp")
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(top_sites["Site S1"].astype(str), top_sites["nb_abp"],
               color="#e07b54", height=0.7)
ax.bar_label(bars, fmt="%d ABPs", padding=3, fontsize=8)
ax.set_xlabel("Nb ABPs distincts")
ax.set_title("Sites S1 partagés par le plus d'ABPs (compétition)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

/var/folders/hv/x060xf8501g94gz9j0y0wqxm0000gn/T/ipykernel_7797/3084347798.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Réseau de compétition : ABPs reliés par sites S1 partagés
import networkx as nx

G = nx.Graph()
for _, row in compet_multi.iterrows():
    abps = row["abps"]
    site = row["Site S1"]
    for i, a in enumerate(abps):
        for b in abps[i+1:]:
            if G.has_edge(a, b):
                G[a][b]["weight"] += 1
                G[a][b]["sites"].append(site)
            else:
                G.add_edge(a, b, weight=1, sites=[site])

print(f"Réseau de compétition : {G.number_of_nodes()} ABPs · {G.number_of_edges()} paires")

# Paires les plus en compétition (partageant le plus de sites S1)
edges_sorted = sorted(G.edges(data=True), key=lambda e: e[2]["weight"], reverse=True)
print("\nTop 10 paires d'ABPs en compétition (nb sites S1 partagés) :")
for a, b, d in edges_sorted[:10]:
    print(f"  {a!r:40s} ↔ {b!r:40s}  → {d['weight']} sites : {d['sites']}")

Réseau de compétition : 39 ABPs · 65 paires

Top 10 paires d'ABPs en compétition (nb sites S1 partagés) :
  'Vibrio VopV'                            ↔ 'VopV'                                    → 3 sites : ['6685_259', '6685_258', '6685_257']
  'Adducin 1'                              ↔ 'Beta-adducin'                            → 3 sites : ['6685_152', '6685_151', '6685_150']
  'Myosin-6'                               ↔ 'Myosin heavy chain 4'                    → 2 sites : ['6685_193', '6685_97']
  'Alpha-catenin-like protein hmp-1'       ↔ 'Catenin alpha-1'                         → 2 sites : ['6685_0', '6685_141']
  'Cofilin-1'                              ↔ 'Cofilin-2'                               → 2 sites : ['6685_0', '6685_21']
  'Cofilin-1'                              ↔ 'Inverted formin-2'                       → 2 sites : ['6685_0', '6685_200']
  'Inverted formin-2'                      ↔ 'Protein diaphanous homolog 1'            → 2 sites : ['6685_200', '6685_211']
  'Epididy

In [9]:
# Visualisation réseau (composantes principales)
# Filtrer les ABPs avec au moins 1 compétiteur
fig, ax = plt.subplots(figsize=(12, 9))

weights = [G[u][v]["weight"] for u, v in G.edges()]
w_max   = max(weights) if weights else 1

pos = nx.spring_layout(G, k=2.5, seed=42)
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=300,
                       node_color="#4a90d9", alpha=0.85)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=7)
nx.draw_networkx_edges(
    G, pos, ax=ax,
    width=[1 + 4 * w / w_max for w in weights],
    edge_color="#e07b54", alpha=0.6,
)
# Légende épaisseur
for w, lbl in [(1, "1 site"), (w_max // 2, f"{w_max//2} sites"), (w_max, f"{w_max} sites")]:
    ax.plot([], [], color="#e07b54",
            linewidth=1 + 4 * w / w_max, label=lbl)
ax.legend(title="Sites S1 partagés", fontsize=8, loc="upper left")
ax.set_title("Réseau de compétition ABP — arêtes = sites S1 partagés", fontsize=12)
ax.axis("off")
plt.tight_layout()
plt.show()

/var/folders/hv/x060xf8501g94gz9j0y0wqxm0000gn/T/ipykernel_7797/4161271737.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Détail par site S1 : quels ABPs sont en compétition ?

In [10]:
# Sélectionner un site S1 pour voir le détail des interactions par ABP
SITE_S1 = "6685_0"  # ← changer ici

site_data = hetero_m[
    hetero_m["s1_binding_site_cluster_data_70"] == SITE_S1
].copy()

detail = (
    site_data.groupby("protein")
    .agg(
        nb_inter=("subunit_1", "count"),
        nb_pdb=("_pdb", "nunique"),
        nb_c70=("cluster_data_70", "nunique"),
        c70_clusters=("cluster_data_70",
                      lambda x: ", ".join(sorted(x.dropna().astype(str).unique()))),
        area_moy=("area", "mean"),
        contacts_moy=("number_of_contacts", "mean"),
    )
    .reset_index()
    .sort_values("nb_inter", ascending=False)
)

print(f"Site S1 : {SITE_S1} — {len(detail)} ABPs en compétition")
detail

Site S1 : 6685_0 — 5 ABPs en compétition


,protein,nb_inter,nb_pdb,nb_c70,c70_clusters,area_moy,contacts_moy
3,Cofilin-2,42,10,1,0_30351_1,1082.097869,91.357143
2,Cofilin-1,27,8,1,0_30351_1,1007.246124,89.148148
1,Catenin alpha-1,7,2,2,"0_36172_1, 0_39640_1",847.528095,76.000000
0,Alpha-catenin-like protein hmp-1,5,1,1,0_65066_1,718.058249,59.000000
4,Inverted formin-2,2,1,1,0_74512_0,407.355210,37.000000


In [11]:
# Charger résidus d'interface et interactions pour relier positions → ABP
df3_nb    = pd.read_csv("../data/filtered/details/3.interface_residues.csv")
df_int_nb = pd.read_csv("../data/filtered/details/1.interactions.csv")

df3_nb["residue_number_canon_mafft"] = pd.to_numeric(
    df3_nb["residue_number_canon_mafft"], errors="coerce")
df3_nb["buried_ASA_percent"] = pd.to_numeric(
    df3_nb["buried_ASA_percent"].astype(str).str.replace("%", "", regex=False),
    errors="coerce")

# Relier les interactions du site S1 aux interaction_ids via 1.interactions.csv
site_with_ids = site_data.merge(
    df_int_nb[["interaction_id", "chain_A_id", "chain_B_id"]],
    left_on=["subunit_1", "subunit_2"],
    right_on=["chain_A_id", "chain_B_id"],
    how="inner"
)

# Total d'interactions par ABP (dénominateur global — absences = 0)
_n_inter_total = (
    site_with_ids.groupby("protein")["interaction_id"].nunique()
    .rename("n_total")
)

# Garder seulement les résidus de la chaîne actine (S1)
_id_to_actin = site_with_ids.set_index("interaction_id")["chain_A_id"].str.lower()
_id_to_prot  = site_with_ids.set_index("interaction_id")["protein"]

df3_site = df3_nb[df3_nb["interaction_id"].isin(site_with_ids["interaction_id"])].copy()
df3_site["_actin_ch"] = df3_site["interaction_id"].map(_id_to_actin)
df3_site = df3_site[df3_site["chain"].str.lower() == df3_site["_actin_ch"]].copy()
df3_site["protein"] = df3_site["interaction_id"].map(_id_to_prot)
df3_site = df3_site.dropna(subset=["residue_number_canon_mafft", "protein"])
df3_site["canon"] = df3_site["residue_number_canon_mafft"].astype(int)

print(f"{df3_site['canon'].nunique()} positions canoniques · {df3_site['protein'].nunique()} ABPs")
print("Nb interactions par ABP (dénominateur) :")
print(_n_inter_total.to_dict())

# Somme ASA par (protein, canon), puis division par le total d'interactions
footprint_sum = (
    df3_site.groupby(["protein", "canon"])["buried_ASA_percent"]
    .sum()
    .reset_index()
    .rename(columns={"buried_ASA_percent": "asa_sum"})
)
footprint_sum = footprint_sum.join(_n_inter_total, on="protein")
footprint_sum["asa_mean"] = footprint_sum["asa_sum"] / footprint_sum["n_total"]

pivot_fp = footprint_sum.pivot_table(
    index="canon", columns="protein",
    values="asa_mean", fill_value=0
)
pivot_fp.columns.name = "ABP"

fig_fp, ax_fp = plt.subplots(
    figsize=(max(7, pivot_fp.shape[1] * 1.8), max(6, pivot_fp.shape[0] * 0.14))
)
sns.heatmap(
    pivot_fp,
    ax=ax_fp,
    cmap="YlOrRd",
    mask=pivot_fp == 0,
    linewidths=0,
    vmin=0,
    cbar_kws={"label": "% ASA buried moyen (/ total interactions)"},
)
ax_fp.set_title(
    f"Empreinte des ABPs sur l'actine — site S1 {SITE_S1} "
    f"({pivot_fp.shape[1]} ABPs, {pivot_fp.shape[0]} positions)",
    fontsize=12,
)
ax_fp.set_xlabel("ABP", fontsize=10)
ax_fp.set_ylabel("Position canonique actine (MAFFT)", fontsize=10)
ax_fp.tick_params(axis="x", rotation=45, labelsize=8)
ax_fp.tick_params(axis="y", labelsize=7)
plt.tight_layout()
plt.show()

76 positions canoniques · 5 ABPs
Nb interactions par ABP (dénominateur) :
{'Alpha-catenin-like protein hmp-1': 5, 'Catenin alpha-1': 7, 'Cofilin-1': 27, 'Cofilin-2': 42, 'Inverted formin-2': 2}


/var/folders/hv/x060xf8501g94gz9j0y0wqxm0000gn/T/ipykernel_7797/499598403.py:78: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. PDB sans ABP — filaments d'actine pure

In [12]:
pdbs_with_abp = set(df_pp[~df_pp["is_actin"]]["pdb_id"])
all_pdbs      = set(df_all["_pdb"])
pdbs_no_abp   = all_pdbs - pdbs_with_abp

print(f"PDB sans ABP : {len(pdbs_no_abp)} / {len(all_pdbs)} au total")

homo_no_abp = df_all[
    df_all["s1_actine"] & df_all["s2_actine"] &
    df_all["_pdb"].isin(pdbs_no_abp) &
    df_all["cluster_data_70"].notna()
].copy()

homo_no_abp["Binding sites"] = homo_no_abp.apply(
    lambda r: " × ".join(sorted([
        str(r["s1_binding_site_cluster_data_70"]),
        str(r["s2_binding_site_cluster_data_70"])
    ])), axis=1
)

summary_no_abp = (
    homo_no_abp.groupby(["cluster_data_70", "Binding sites"])
    .agg(nb_pdb=("_pdb", "nunique"), nb_inter=("_pdb", "count"))
    .reset_index()
    .sort_values("nb_pdb", ascending=False)
)
summary_no_abp["%PDB"]   = (summary_no_abp["nb_pdb"]   / len(pdbs_no_abp) * 100).round(1)
summary_no_abp["%inter"] = (summary_no_abp["nb_inter"] / len(homo_no_abp)  * 100).round(1)

summary_no_abp

PDB sans ABP : 60 / 145 au total


,cluster_data_70,Binding sites,nb_pdb,nb_inter,%PDB,%inter
0,0_7797_0,6685_1 × 6685_2,60,258,100.0,53.8
2,0_7797_1,6685_3 × 6685_4,57,199,95.0,41.5
1,0_7797_0,6685_109 × 6685_2,3,10,5.0,2.1
5,0_7797_45,6685_3 × 6685_4,2,8,3.3,1.7
3,0_7797_10,6685_1 × 6685_2,1,2,1.7,0.4
4,0_7797_32,6685_3 × 6685_4,1,3,1.7,0.6


In [13]:
# Bar chart clusters actine-actine dans PDB sans ABP
s_na = summary_no_abp.sort_values("nb_inter")
lbl  = s_na["cluster_data_70"].astype(str) + "  [" + s_na["Binding sites"] + "]"

fig, ax = plt.subplots(figsize=(9, max(3, len(s_na) * 0.45)))
bars = ax.barh(lbl, s_na["nb_inter"], color="#7b9ec2", height=0.6)
ax.bar_label(bars, fmt="%d", padding=3, fontsize=8)
ax.set_xlabel("Nb interactions")
ax.set_title("Interactions actine-actine dans les PDB sans ABP")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

/var/folders/hv/x060xf8501g94gz9j0y0wqxm0000gn/T/ipykernel_7797/655903481.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
